# Volume Rendering with ImageData

When your xarray data has **uniform spacing** on all axes, `pyvista-xarray`
automatically creates a `pyvista.ImageData` instead of a `RectilinearGrid`.
This enables VTK's GPU-accelerated volume rendering, which is significantly
faster than the software-based rendering required for `RectilinearGrid`.

This notebook demonstrates:
1. Automatic ImageData detection for uniformly-spaced grids
2. Volume rendering of 3D fluorescence microscopy data
3. Comparing ImageData vs RectilinearGrid representations

In [ ]:
import numpy as np
import pyvista as pv
import xarray as xr

import pvxarray  # noqa: F401 - registers the .pyvista accessor

## The cells3d Dataset

The xarray tutorial dataset `cells3d` contains 3D fluorescence microscopy
images of cells with two channels: nuclei and membrane. The spatial
coordinates (`x`, `y`, `z`) have uniform spacing of ~0.26 microns,
making it a perfect candidate for `ImageData`.

In [ ]:
ds = xr.tutorial.load_dataset("cells3d")
ds

In [ ]:
# Select the nuclei channel
da = ds.images.sel(c="nuclei")
print(f"Shape: {da.shape}")
print(f"Size: {da.nbytes / 1024 / 1024:.1f} MB")
print()

# Verify uniform spacing
for coord in ["x", "y", "z"]:
    vals = da[coord].values
    diffs = np.diff(vals)
    print(
        f"{coord}: {len(vals)} points, spacing = {diffs[0]:.4f}, uniform = {np.allclose(diffs, diffs[0])}"
    )

## Creating the Mesh

When we call `.pyvista.mesh()`, the uniform spacing is automatically
detected and an `ImageData` is returned instead of a `RectilinearGrid`.

In [ ]:
nuclei = da.pyvista.mesh(x="x", y="y", z="z")
print(f"Mesh type: {type(nuclei).__name__}")
print(f"Dimensions: {nuclei.dimensions}")
print(f"Spacing: {nuclei.spacing}")
print(f"Origin: {nuclei.origin}")
nuclei

## Volume Rendering

With `ImageData`, PyVista's `add_volume` uses VTK's GPU-accelerated
`vtkGPUVolumeRayCastMapper`, which provides fast, high-quality volume
rendering. This would not work as efficiently with a `RectilinearGrid`,
which would require conversion to an `UnstructuredGrid` first.

In [ ]:
pl = pv.Plotter()
pl.add_volume(nuclei, clim=(0, 30000), opacity="sigmoid")
pl.enable_terrain_style()
pl.show()

## Both Channels

We can visualize both the nuclei and membrane channels together
by adding two volumes to the same plotter with different colormaps.

In [ ]:
membrane = ds.images.sel(c="membrane").pyvista.mesh(x="x", y="y", z="z")

pl = pv.Plotter()
pl.add_volume(nuclei, clim=(0, 30000), opacity="sigmoid", cmap="Blues")
pl.add_volume(membrane, clim=(0, 40000), opacity="sigmoid", cmap="Oranges")
pl.enable_terrain_style()
pl.show()

## How Detection Works

The `ImageData` optimization is automatic: when all coordinate axes have
uniform spacing (checked via `np.allclose`), `pyvista-xarray` creates an
`ImageData`. Otherwise, it falls back to `RectilinearGrid`.

Here we compare both cases:

In [ ]:
# Uniform spacing -> ImageData
x_uniform = np.linspace(0, 10, 50)
y_uniform = np.linspace(0, 5, 25)
data = np.random.randn(50, 25)

da_uniform = xr.DataArray(
    data,
    dims=["x", "y"],
    coords={"x": x_uniform, "y": y_uniform},
    name="temperature",
)
mesh_uniform = da_uniform.pyvista.mesh(x="x", y="y")
print(f"Uniform spacing -> {type(mesh_uniform).__name__}")

# Non-uniform spacing -> RectilinearGrid
x_nonuniform = np.array([0, 1, 3, 6, 10, 15, 21, 28, 36, 45])
y_nonuniform = np.linspace(0, 5, 25)
data2 = np.random.randn(10, 25)

da_nonuniform = xr.DataArray(
    data2,
    dims=["x", "y"],
    coords={"x": x_nonuniform, "y": y_nonuniform},
    name="temperature",
)
mesh_nonuniform = da_nonuniform.pyvista.mesh(x="x", y="y")
print(f"Non-uniform spacing -> {type(mesh_nonuniform).__name__}")

## Roundtrip: ImageData to xarray and Back

The `pyvista_to_xarray` function also handles `ImageData`,
generating coordinate arrays from the origin, spacing, and dimensions.
This enables clean roundtrips between PyVista and xarray.

In [ ]:
from pvxarray import pyvista_to_xarray

# Convert ImageData mesh back to xarray
ds_roundtrip = pyvista_to_xarray(nuclei)
print("Roundtrip dataset:")
print(ds_roundtrip)
print()

# Create mesh again - should still be ImageData
mesh_roundtrip = ds_roundtrip["images"].pyvista.mesh(x="x", y="y", z="z")
print(f"Roundtrip mesh type: {type(mesh_roundtrip).__name__}")
print(f"Origin matches: {np.allclose(nuclei.origin, mesh_roundtrip.origin)}")
print(f"Spacing matches: {np.allclose(nuclei.spacing, mesh_roundtrip.spacing)}")